## 1. Import Library & Setup

Memuat pustaka yang dibutuhkan untuk web scraping, pengolahan data tabular, dan spasial.

In [1]:
import requests
import pandas as pd
import geopandas as gpd

from shapely.geometry import shape
from tqdm.auto import tqdm
from pathlib import Path

c:\Users\prata\Documents\Kuliah\CodeLabs\COMPFEST\2026\Final Study Case\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Konfigurasi

Mengatur parameter utama seperti URL MapServer BIG, direktori output, dan periode penelitian.

In [2]:
# =========================================================
# CONFIGURATION
# =========================================================

BASE_URL = (
    "https://kspservices.big.go.id/satupeta/rest/services/"
    "PUBLIK/SUMBER_DAYA_ALAM_DAN_LINGKUNGAN/MapServer/6"
)

QUERY_URL = f"{BASE_URL}/query"

OUTPUT_DIR = Path("../datasets/gambut")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Periode penelitian
START_DATE = "2024-08-01"
END_DATE   = "2026-05-31"

# Batch size pengambilan data
BATCH_SIZE = 100

# CRS
TARGET_CRS = "EPSG:4326"

## 3. Pemeriksaan Metadata

Mengambil informasi metadata layer (jenis geometri, CRS) serta daftar `objectid` poligon lahan gambut.

In [3]:
response = requests.get(
    BASE_URL,
    params={"f": "json"},
    timeout=60
)

response.raise_for_status()

metadata = response.json()

print("Nama layer :", metadata["name"])
print("Tipe       :", metadata["type"])
print("Geometry   :", metadata["geometryType"])
print("CRS        :", metadata["sourceSpatialReference"]["wkid"])
print("Max record :", metadata["maxRecordCount"])

Nama layer : Peta Lahan Gambut
Tipe       : Feature Layer
Geometry   : esriGeometryPolygon
CRS        : 4326
Max record : 1000


In [4]:
fields = metadata["fields"]

field_df = pd.DataFrame([
    {
        "name": field["name"],
        "type": field["type"],
        "alias": field.get("alias")
    }
    for field in fields
])

field_df

,name,type,alias
0,objectid,esriFieldTypeOID,objectid
1,wadmpu,esriFieldTypeString,wadmpu
2,landform,esriFieldTypeString,landform
3,bhnindk,esriFieldTypeString,bhnindk
4,relief,esriFieldTypeString,relief
5,usda1,esriFieldTypeString,usda1
6,usda2,esriFieldTypeString,usda2
7,usda3,esriFieldTypeString,usda3
8,fcode,esriFieldTypeString,fcode
9,metadata,esriFieldTypeString,metadata


In [5]:
params = {
    "where": "1=1",
    "returnIdsOnly": "true",
    "f": "json"
}

response = requests.get(
    QUERY_URL,
    params=params,
    timeout=60
)

response.raise_for_status()

id_data = response.json()

object_ids = id_data.get("objectIds", [])

object_ids = sorted(object_ids)

print("Total feature:", len(object_ids))
print("ID pertama  :", object_ids[:10])

Total feature: 146
ID pertama  : [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]


## 4. Proses Unduh Data Lahan Gambut

Mengunduh fitur poligon dari server secara *batch* untuk menghindari batas maksimum dan *timeout*.

In [6]:
def get_features_by_ids(object_ids_batch):

    params = {
        "objectIds": ",".join(map(str, object_ids_batch)),
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": 4326,
        "f": "geojson"
    }

    response = requests.get(
        QUERY_URL,
        params=params,
        timeout=120
    )

    response.raise_for_status()

    data = response.json()

    if "error" in data:
        raise RuntimeError(data["error"])

    return data.get("features", [])

In [7]:
all_features = []

for i in tqdm(
    range(0, len(object_ids), BATCH_SIZE),
    desc="Downloading BIG peatland polygons"
):
    
    batch_ids = object_ids[i:i + BATCH_SIZE]

    features = get_features_by_ids(batch_ids)

    all_features.extend(features)

print("Total feature berhasil diambil:", len(all_features))

Total feature berhasil diambil: 146


In [8]:
gambut = gpd.GeoDataFrame.from_features(
    all_features,
    crs=TARGET_CRS
)

print(gambut.shape)

(146, 13)


## 5. Pemeriksaan dan Perbaikan Geometri

Memeriksa data tabular dan memperbaiki *invalid geometry* (jika ada) menggunakan `.make_valid()`.

In [9]:
gambut.head()

,geometry,objectid,wadmpu,landform,bhnindk,relief,usda1,usda2,usda3,fcode,metadata,srs_id,namobj
0,"MULTIPOLYGON (((112.63986 -2.09699, 112.63881 ...",1,Kalimantan,Rawa Belakang Sungai Meander (Backswamp),Endapan organik,Datar,Terric Haplosaprists,-,-,None,None,None,None
1,"MULTIPOLYGON (((116.75025 0.07074, 116.74949 0...",2,Kalimantan,Rawa Belakang Sungai Meander (Backswamp),Endapan organik,Datar,Terric Haplosaprists,Terric Haplohemists,-,None,None,None,None
2,"MULTIPOLYGON (((112.16631 -2.70538, 112.16604 ...",3,Kalimantan,Rawa Belakang Sungai Meander (Backswamp),Endapan organik dan mineral,Datar,Terric Haplosaprists,Typic Endoaquents,-,None,None,None,None
3,"POLYGON ((116.33817 -0.17549, 116.33804 -0.175...",4,Kalimantan,Rawa Belakang Sungai Meander (Backswamp),Endapan organik dan mineral,Datar,Terric Haplosaprists,Humaqueptic Endoaquents,-,None,None,None,None
4,"MULTIPOLYGON (((114.80213 -2.4477, 114.80246 -...",5,Kalimantan,Rawa Belakang Sungai Meander (Backswamp),Endapan organik,Datar,Terric Haplohemists,Terric Sulfihemists,-,None,None,None,None


In [10]:
print(gambut.crs)

EPSG:4326


In [11]:
if "shape" in gambut.columns:
    gambut = gambut.drop(columns=["shape"])

In [12]:
gambut.columns

Index(['geometry', 'objectid', 'wadmpu', 'landform', 'bhnindk', 'relief',
       'usda1', 'usda2', 'usda3', 'fcode', 'metadata', 'srs_id', 'namobj'],
      dtype='str')

In [13]:
print("Geometry kosong :", gambut.geometry.isna().sum())
print("Geometry invalid:", (~gambut.geometry.is_valid).sum())

Geometry kosong : 0
Geometry invalid: 0


In [14]:
invalid_mask = ~gambut.geometry.is_valid

gambut.loc[invalid_mask, "geometry"] = (
    gambut.loc[invalid_mask, "geometry"].make_valid()
)

In [15]:
print(
    "Geometry invalid setelah perbaikan:",
    (~gambut.geometry.is_valid).sum()
)

Geometry invalid setelah perbaikan: 0


## 6. Identifikasi dan Klasifikasi Lahan Gambut

Menentukan poligon mana yang merupakan lahan gambut, serta mengelompokkan *peat depth* (kedalaman gambut) dan *peat type* (tipe gambut).

In [16]:
gambut["is_peatland"] = gambut["landform"].notna()

In [18]:
gambut["is_peatland"] = (
    gambut["landform"]
    .fillna("")
    .str.contains("Gambut", case=False, na=False)
)

In [19]:
gambut["is_peatland"].value_counts()

is_peatland
True     100
False     46
Name: count, dtype: int64

In [20]:
gambut.groupby("is_peatland")["objectid"].count()

is_peatland
False     46
True     100
Name: objectid, dtype: int64

In [21]:
def classify_depth(landform):

    if pd.isna(landform):
        return "Non-gambut"

    text = str(landform)

    if "> 700 cm" in text:
        return ">700 cm"
    elif "500 cm - < 700 cm" in text:
        return "500-700 cm"
    elif "300 cm - < 500 cm" in text:
        return "300-500 cm"
    elif "200 cm - < 300 cm" in text:
        return "200-300 cm"
    elif "100 cm - < 200 cm" in text:
        return "100-200 cm"
    elif "50 cm - < 100 cm" in text:
        return "50-100 cm"
    else:
        return "Tidak diketahui"

In [22]:
gambut["peat_depth"] = gambut["landform"].apply(classify_depth)

In [23]:
gambut["peat_depth"].value_counts()

peat_depth
Tidak diketahui    46
100-200 cm         25
300-500 cm         21
200-300 cm         21
50-100 cm          20
500-700 cm         12
>700 cm             1
Name: count, dtype: int64

In [24]:
def classify_peat_type(landform):

    if pd.isna(landform):
        return "Non-gambut"

    text = str(landform)

    if "Topogen Air Payau" in text:
        return "Topogen Air Payau"

    elif "Topogen Air Tawar" in text:
        return "Topogen Air Tawar"

    elif "Tepi Kubah Gambut" in text:
        return "Tepi Kubah Gambut"

    elif "Kubah Gambut" in text:
        return "Kubah Gambut"

    else:
        return "Lainnya"

In [25]:
gambut["peat_type"] = gambut["landform"].apply(
    classify_peat_type
)

## 7. Penyimpanan Data Spasial

Memilih kolom penting dan menyimpannya dalam format `.geojson` dan `.gpkg` (GeoPackage).

In [26]:
final_columns = [
    "objectid",
    "wadmpu",
    "landform",
    "peat_type",
    "peat_depth",
    "is_peatland",
    "bhnindk",
    "relief",
    "usda1",
    "usda2",
    "usda3",
    "fcode",
    "namobj",
    "metadata",
    "srs_id",
    "geometry"
]

gambut_final = gambut[
    [col for col in final_columns if col in gambut.columns]
].copy()

In [27]:
geojson_path = OUTPUT_DIR / "peta_lahan_gambut_kalimantan.geojson"

gambut_final.to_file(
    geojson_path,
    driver="GeoJSON"
)

print(f"Saved: {geojson_path}")

Saved: ..\datasets\gambut\peta_lahan_gambut_kalimantan.geojson


In [28]:
gpkg_path = OUTPUT_DIR / "peta_lahan_gambut_kalimantan.gpkg"

gambut_final.to_file(
    gpkg_path,
    layer="peatland",
    driver="GPKG"
)

print(f"Saved: {gpkg_path}")

Saved: ..\datasets\gambut\peta_lahan_gambut_kalimantan.gpkg


In [29]:
print("Jumlah polygon :", len(gambut_final))
print("CRS            :", gambut_final.crs)
print("Geometry       :", gambut_final.geom_type.value_counts().to_dict())
print("Invalid geometry:", (~gambut_final.geometry.is_valid).sum())

Jumlah polygon : 146
CRS            : EPSG:4326
Geometry       : {'MultiPolygon': 109, 'Polygon': 37}
Invalid geometry: 0


In [30]:
gambut_final[
    [
        "landform",
        "peat_type",
        "peat_depth",
        "is_peatland"
    ]
].head(20)

,landform,peat_type,peat_depth,is_peatland
0,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
1,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
2,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
3,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
4,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
5,Rawa Belakang Sungai Meander (Backswamp),Lainnya,Tidak diketahui,False
6,Tepi Kubah Gambut (50 cm - < 100 cm),Tepi Kubah Gambut,50-100 cm,True
7,Gambut Topogen Air Tawar (50 cm - < 100 cm),Topogen Air Tawar,50-100 cm,True
8,Gambut Topogen Air Tawar (50 cm - < 100 cm),Topogen Air Tawar,50-100 cm,True
9,Gambut Topogen Air Tawar (50 cm - < 100 cm),Topogen Air Tawar,50-100 cm,True


In [31]:
gambut_final["is_peatland"].value_counts()

is_peatland
True     100
False     46
Name: count, dtype: int64

In [32]:
gambut_area = gambut_final.to_crs(
    "ESRI:102025"
)

In [33]:
gambut_area["area_km2"] = (
    gambut_area.geometry.area / 1_000_000
)

In [34]:
total_area = gambut_area["area_km2"].sum()

print(f"Total area polygon: {total_area:,.2f} km²")

Total area polygon: 45,281.91 km²


In [35]:
depth_summary = (
    gambut_area[gambut_area["is_peatland"]]
    .groupby("peat_depth", as_index=False)
    ["area_km2"]
    .sum()
    .sort_values("area_km2", ascending=False)
)

depth_summary

,peat_depth,area_km2
0,100-200 cm,12998.198855
1,200-300 cm,10795.977701
2,300-500 cm,7763.727343
3,50-100 cm,5751.707072
4,500-700 cm,5061.924425
5,>700 cm,418.017752


In [36]:
START_DATE = "2024-08-01"
END_DATE   = "2026-05-31"

## 8. Integrasi dengan Data Titik Panas (Hotspot)

Memuat data historis FIRMS dan melakukan *Spatial Join* (`sjoin`) untuk memetakan titik panas ke dalam area lahan gambut.

In [38]:
firms = pd.read_csv('../datasets/processed/fireline_hotspot_clean.csv')

firms["acq_date"] = pd.to_datetime(
    firms["acq_date"]
)

firms = firms[
    (firms["acq_date"] >= START_DATE) &
    (firms["acq_date"] <= END_DATE)
].copy()

In [39]:
hotspot = gpd.GeoDataFrame(
    firms,
    geometry=gpd.points_from_xy(
        firms["longitude"],
        firms["latitude"]
    ),
    crs="EPSG:4326"
)

In [40]:
hotspot_gambut = gpd.sjoin(
    hotspot,
    gambut_final[
        [
            "objectid",
            "landform",
            "peat_type",
            "peat_depth",
            "is_peatland",
            "geometry"
        ]
    ],
    how="left",
    predicate="within"
)

In [41]:
hotspot_gambut["peatland_status"] = (
    hotspot_gambut["is_peatland"]
    .map({
        True: "Gambut",
        False: "Non-Gambut"
    })
    .fillna("Di luar area terpetakan")
)

In [42]:
hotspot_gambut["peatland_status"].value_counts()

peatland_status
Di luar area terpetakan    55780
Gambut                      5600
Non-Gambut                   203
Name: count, dtype: int64

## 9. Analisis Data Hotspot di Lahan Gambut

Menghitung statistik bulanan dan intensitas *Fire Radiative Power* (FRP) pada area gambut vs non-gambut.

In [43]:
hotspot_gambut["month"] = (
    hotspot_gambut["acq_date"]
    .dt.to_period("M")
    .astype(str)
)

In [44]:
monthly = (
    hotspot_gambut
    .groupby(
        ["month", "peatland_status"]
    )
    .size()
    .reset_index(name="hotspot_count")
)

In [45]:
frp_summary = (
    hotspot_gambut
    .groupby("peatland_status")
    .agg(
        hotspot_count=("frp", "count"),
        mean_frp=("frp", "mean"),
        median_frp=("frp", "median"),
        max_frp=("frp", "max")
    )
    .reset_index()
)

frp_summary

,peatland_status,hotspot_count,mean_frp,median_frp,max_frp
0,Di luar area terpetakan,55780,10.770348,6.30,954.79
1,Gambut,5600,8.713630,4.68,652.45
2,Non-Gambut,203,6.711823,5.03,33.51


In [46]:
depth_fire = (
    hotspot_gambut[
        hotspot_gambut["is_peatland"] == True
    ]
    .groupby("peat_depth")
    .agg(
        hotspot_count=("frp", "count"),
        mean_frp=("frp", "mean"),
        median_frp=("frp", "median"),
        max_frp=("frp", "max")
    )
    .reset_index()
)

depth_fire

,peat_depth,hotspot_count,mean_frp,median_frp,max_frp
0,100-200 cm,2497,8.836556,4.640,550.26
1,200-300 cm,1270,8.955677,4.495,349.42
2,300-500 cm,1187,8.158905,4.650,652.45
3,50-100 cm,337,10.092582,5.480,201.77
4,500-700 cm,309,7.352492,4.710,116.89


In [47]:
output_fire = OUTPUT_DIR / (
    "firms_kalimantan_gambut_"
    "20240801_20260531.gpkg"
)

hotspot_gambut.to_file(
    output_fire,
    layer="hotspot_gambut",
    driver="GPKG"
)

In [48]:
csv_output = OUTPUT_DIR / (
    "firms_kalimantan_gambut_"
    "20240801_20260531.csv"
)

hotspot_gambut.drop(
    columns="geometry"
).to_csv(
    csv_output,
    index=False
)